# MDC Dataset — preprocessing & window tensors

**Part 1:** download/clean flows → save `mdc_preprocessed.parquet` (+ scaler).  
**Part 2:** 15-second buckets → sliding windows → `windows.npz` for modelling.

**Runs on Google Colab or locally.** Default load uses [Kaggle Hub](https://www.kaggle.com/datasets/yigitsever/misuse-detection-in-containers-dataset). Use Colab Secrets `KAGGLE_USERNAME` / `KAGGLE_KEY`, or `~/.kaggle/kaggle.json`.


## 0. Imports & Config

In [ ]:
import sys
import os
import subprocess

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

try:
    import google.colab  # noqa: F401
    _IN_COLAB = True
except ImportError:
    _IN_COLAB = False

if _IN_COLAB:
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', 'kagglehub[pandas-datasets]', 'joblib'],
        check=False,
    )
    OUTPUT_DIR = os.path.join(os.getcwd(), 'data', 'processed')
else:
    OUTPUT_DIR = os.path.normpath('../data/processed')

os.makedirs(OUTPUT_DIR, exist_ok=True)

DATA_PATH_LOCAL = os.path.normpath('../data/raw/MDC dataset.csv')

# ── Thresholds ─────────────────────────────────────────────────────────────
MIN_FLOWS     = 10_000
GAP_THRESHOLD = 60

# ── Window config (Part 1 feasibility check only — Part 2 overrides these) ─
WINDOW_SIZE = 500
STRIDE      = 100

# ── Feature selection thresholds ───────────────────────────────────────────
VARIANCE_THRESHOLD = 0.01
CORR_THRESHOLD     = 0.95

# ── Columns to always drop ─────────────────────────────────────────────────
DROP_COLS = [
    'Flow ID', 'Dst IP', 'Timestamp',
    'Fwd URG Flags', 'Bwd URG Flags',
    'URG Flag Count', 'CWR Flag Count', 'ECE Flag Count'
]

print('Config loaded.')
print(f'  IN_COLAB           = {_IN_COLAB}')
print(f'  OUTPUT_DIR         = {OUTPUT_DIR}')
print(f'  MIN_FLOWS          = {MIN_FLOWS:,}')
print(f'  GAP_THRESHOLD      = {GAP_THRESHOLD}s')
print(f'  VARIANCE_THRESHOLD = {VARIANCE_THRESHOLD}')
print(f'  CORR_THRESHOLD     = {CORR_THRESHOLD}')

Config loaded.
  IN_COLAB           = True
  OUTPUT_DIR         = /content/data/processed
  MIN_FLOWS          = 10,000
  GAP_THRESHOLD      = 60s
  VARIANCE_THRESHOLD = 0.01
  CORR_THRESHOLD     = 0.95


## 1. Load Raw Data

In [ ]:
from pathlib import Path

import kagglehub
from kagglehub import KaggleDatasetAdapter

KAGGLE_DATASET = 'yigitsever/misuse-detection-in-containers-dataset'
KAGGLE_CSV_RELATIVE = 'MDC dataset.csv'

if _IN_COLAB:
    try:
        from google.colab import userdata

        u = userdata.get('KAGGLE_USERNAME')
        k = userdata.get('KAGGLE_KEY')
        if u:
            os.environ['KAGGLE_USERNAME'] = u
        if k:
            os.environ['KAGGLE_KEY'] = k
    except Exception:
        pass

try:
    df = kagglehub.load_dataset(
        KaggleDatasetAdapter.PANDAS,
        KAGGLE_DATASET,
        KAGGLE_CSV_RELATIVE,
    )
except Exception:
    root = Path(kagglehub.dataset_download(KAGGLE_DATASET))
    exact = root / KAGGLE_CSV_RELATIVE
    if exact.exists():
        df = pd.read_csv(exact)
    else:
        candidates = sorted(root.rglob('*.csv'), key=lambda p: p.stat().st_size, reverse=True)
        if not candidates:
            raise FileNotFoundError(f'No .csv found under downloaded dataset root: {root}')
        df = pd.read_csv(candidates[0])
        print(f'Using CSV: {candidates[0]} (expected {KAGGLE_CSV_RELATIVE!r}; adjust cell if layout changed)')

# --- Alternate: load from disk (kernel cwd = notebook/ for relative path) -----
# df = pd.read_csv(DATA_PATH_LOCAL)

df.columns = df.columns.str.strip()

print(f'Shape  : {df.shape}')
print(f'Rows   : {df.shape[0]:,}')
print(f'Cols   : {df.shape[1]}')
print(f'Labels : {sorted(df["Label"].unique())}')

100%|██████████| 552M/552M [00:04<00:00, 128MB/s]

Extracting files...


Using CSV: /root/.cache/kagglehub/datasets/yigitsever/misuse-detection-in-containers-dataset/versions/1/dataset.csv (expected 'MDC dataset.csv'; adjust cell if layout changed)
Shape  : (3231475, 87)
Rows   : 3,231,475
Cols   : 87
Labels : [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11)]


## 2. Container Filtering
Group by `Src IP`. Keep only containers with flows ≥ `MIN_FLOWS`.

In [ ]:
# Count flows per container
container_counts = df['Src IP'].value_counts()

print('All containers (Src IP) and their flow counts:')
print(container_counts.to_string())

# Apply threshold
top_containers = container_counts[container_counts >= MIN_FLOWS].index.tolist()

print(f'\n── Threshold: {MIN_FLOWS:,} flows ──')
print(f'Containers kept   : {len(top_containers)}')
print(f'Containers dropped: {len(container_counts) - len(top_containers)}')
print('\nKept containers:')
for ip in top_containers:
    pct = container_counts[ip] / len(df) * 100
    print(f'  {ip:<20} {container_counts[ip]:>10,} flows  ({pct:.2f}%)')

All containers (Src IP) and their flow counts:
Src IP
100.64.0.2         2822554
10.16.0.9           225405
10.16.0.6            53267
10.16.0.5            34400
10.16.0.4            34224
10.16.0.61           21315
10.16.0.49            9072
10.16.0.2             4857
10.16.0.41            3291
144.122.71.18         2835
10.16.0.7             2278
10.16.0.43             349
8.6.0.1                255
10.16.0.45              73
10.16.0.58              67
10.16.0.18              67
10.16.1.16              63
10.16.0.48              60
10.16.0.8               40
10.16.0.52              27
100.64.0.1              22
10.16.0.62              21
10.16.0.66               9
10.16.0.12               8
10.16.0.14               8
10.16.0.63               6
144.122.171.92           6
10.98.54.171             5
10.16.10.199             4
10.16.9.9                4
10.16.10.156             4
10.16.14.229             4
10.16.0.136              4
10.16.14.150             4
10.16.3.2                4
1

In [ ]:
# Filter to top containers only
df = df[df['Src IP'].isin(top_containers)].copy()
df = df.reset_index(drop=True)

print(f'Rows after container filter: {len(df):,}')

Rows after container filter: 3,191,165


In [ ]:
# ── Remove duplicate rows ──────────────────────────────────────────────────
# Network exporters can sometimes emit the same flow record more than once.
# Duplicates in training data bias the autoencoder toward repeated patterns.

before = len(df)
df     = df.drop_duplicates().reset_index(drop=True)

print(f'Duplicate rows removed : {before - len(df):,}')
print(f'Rows remaining         : {len(df):,}')

Duplicate rows removed : 0
Rows remaining         : 3,191,165


## 3. Parse Timestamp & Sort
Sort by `Src IP` then `Timestamp` — required before gap detection.

In [ ]:
# Parse timestamp
df['ts'] = pd.to_datetime(df['Timestamp'], errors='coerce')

failed = df['ts'].isna().sum()
print(f'Parse failures: {failed}')

# Drop unparseable rows
df = df.dropna(subset=['ts'])

# Sort: container first, then time
df = df.sort_values(['Src IP', 'ts']).reset_index(drop=True)

print(f'Rows after sort : {len(df):,}')
print(f'Time range      : {df["ts"].min()}  →  {df["ts"].max()}')

Parse failures: 1
Rows after sort : 3,191,164
Time range      : 2023-03-09 01:01:19.073584  →  2024-05-07 18:40:57.739199


## 4. Long-Gap Detection → Session Assignment
Within each container, gaps > `GAP_THRESHOLD` seconds mark the start of a new **session**.  
A `session_id` column is added — the sliding window must **never cross session boundaries**.

In [ ]:
def assign_sessions(group, threshold):
    """Assign session_id within one container based on time gaps."""
    # Gap between consecutive flows (in seconds)
    gap = group['ts'].diff().dt.total_seconds().fillna(0)
    # Cumsum of gap breaks → session counter per container
    session_num = (gap > threshold).cumsum()
    group['session_id'] = group['Src IP'].astype(str) + '_sess' + session_num.astype(str)
    return group

df = df.groupby('Src IP', group_keys=False).apply(
    assign_sessions, threshold=GAP_THRESHOLD
)

# Summary
session_summary = df.groupby(['Src IP', 'session_id']).size().reset_index(name='flows')
sessions_per_container = session_summary.groupby('Src IP')['session_id'].count()

print(f'Total sessions created : {df["session_id"].nunique()}')
print(f'\nSessions per container:')
print(sessions_per_container.to_string())
print(f'\nSession size stats (flows per session):')
print(session_summary['flows'].describe().round(0))

Total sessions created : 3043

Sessions per container:
Src IP
10.16.0.4      577
10.16.0.5      584
10.16.0.6      503
10.16.0.61    1044
10.16.0.9       57
100.64.0.2     278

Session size stats (flows per session):
count      3043.0
mean       1049.0
std        9022.0
min           1.0
25%           3.0
50%          22.0
75%          60.0
max      240165.0
Name: flows, dtype: float64


In [ ]:
# Show large gaps that triggered session splits
df['time_gap_s'] = df.groupby('Src IP')['ts'].diff().dt.total_seconds()
large_gaps = df[df['time_gap_s'] > GAP_THRESHOLD][['Src IP', 'ts', 'time_gap_s', 'session_id']]

print(f'Session boundary rows (gap > {GAP_THRESHOLD}s): {len(large_gaps)}')
print(large_gaps[['Src IP', 'ts', 'time_gap_s']]
      .sort_values('time_gap_s', ascending=False)
      .head(15)
      .to_string(index=False))

Session boundary rows (gap > 60s): 3037
    Src IP                         ts   time_gap_s
 10.16.0.9 2023-12-06 16:12:25.217228 2.255266e+07
10.16.0.61 2024-04-25 15:29:46.815637 1.765305e+07
 10.16.0.5 2023-09-20 16:50:12.752064 1.589993e+07
 10.16.0.4 2023-09-20 16:50:12.752058 1.589992e+07
 10.16.0.6 2023-09-20 16:50:12.111852 1.589992e+07
100.64.0.2 2023-09-20 16:50:03.509651 1.589991e+07
 10.16.0.5 2024-02-29 22:07:29.965590 7.331760e+06
 10.16.0.4 2024-02-29 22:07:29.966966 7.331749e+06
 10.16.0.9 2024-02-29 22:07:35.358383 7.331734e+06
 10.16.0.6 2024-02-29 22:07:29.513898 7.331728e+06
100.64.0.2 2024-02-29 22:07:27.365545 7.331726e+06
 10.16.0.4 2023-12-04 21:53:28.663229 5.320769e+06
 10.16.0.6 2023-12-04 21:53:28.238311 5.320758e+06
 10.16.0.5 2023-12-04 21:53:28.663220 5.320758e+06
100.64.0.2 2023-12-04 21:53:25.245737 5.320748e+06


## 5. Drop Useless Columns
Remove: identifier columns, raw timestamp, zero-variance features (found in EDA).

In [ ]:
# Save these separately before dropping
meta = df[['Src IP', 'session_id', 'Label', 'ts']].copy()

# Also drop internal helper columns
extra_drop = ['ts', 'time_gap_s', 'Protocol_Name'] if 'Protocol_Name' in df.columns else ['ts', 'time_gap_s']

df_feat = df.drop(columns=DROP_COLS + extra_drop + ['Label', 'session_id'], errors='ignore')

print(f'Features before drop : {df.shape[1]}')
print(f'Features after drop  : {df_feat.shape[1]}')
print(f'Dropped columns      : {DROP_COLS + extra_drop}')

Features before drop : 90
Features after drop  : 78
Dropped columns      : ['Flow ID', 'Dst IP', 'Timestamp', 'Fwd URG Flags', 'Bwd URG Flags', 'URG Flag Count', 'CWR Flag Count', 'ECE Flag Count', 'ts', 'time_gap_s']


## 6. Fix Inf Values
Replace `±inf` with `NaN` — affects `Flow Bytes/s` and `Flow Packets/s` (identified in EDA).

In [ ]:
# Count before
inf_count = np.isinf(df_feat.select_dtypes(include=np.number)).sum()
inf_cols   = inf_count[inf_count > 0]

print(f'Columns with Inf values:')
print(inf_cols.to_string())

# Replace inf → NaN
df_feat = df_feat.replace([np.inf, -np.inf], np.nan)

print(f'\nInf values remaining: {np.isinf(df_feat.select_dtypes(include=np.number)).sum().sum()}')

Columns with Inf values:
Flow Bytes/s        614
Flow Packets/s    41404

Inf values remaining: 0


## 7. Fix Missing Values
Fill NaN with **column median** — robust against outliers.

In [ ]:
missing_before = df_feat.isnull().sum().sum()

# Select only numeric columns
df_feat = df_feat.select_dtypes(include=np.number)

# Fill with column median
col_medians = df_feat.median()
df_feat     = df_feat.fillna(col_medians)

missing_after = df_feat.isnull().sum().sum()

# [LOW-RAM / Colab] Downcast to float32 — ~half the RAM of float64 for 3M+ rows.
df_feat = df_feat.astype(np.float32)

print(f'Missing values : {missing_before:,} → {missing_after}')
print(f'Numeric features kept: {df_feat.shape[1]}')

Missing values : 248,424 → 0
Numeric features kept: 77


In [ ]:
# ── Protocol: one-hot encode (LOW-RAM: avoid get_dummies+concat peak) ─────
# Original pattern duplicated the whole frame during pd.concat.
# Here we .pop('Protocol') and add one binary Proto_* column per observed value.
# [Colab / limited RAM]

import gc

proto = df_feat.pop('Protocol')
pvals = proto.astype(np.float32).to_numpy(copy=False)
proto_cols = []
for u in np.unique(pvals):
    if np.isnan(u):
        continue
    name = f'Proto_{int(u)}'
    df_feat[name] = (pvals == u).astype(np.float32)
    proto_cols.append(name)

del proto, pvals
gc.collect()

print(f'Protocol one-hot cols  : {proto_cols}')
print(f'Features after encode  : {df_feat.shape[1]}')

Protocol one-hot cols  : ['Proto_0', 'Proto_6', 'Proto_17']
Features after encode  : 79


In [ ]:
# ── Near-zero variance: drop uninformative features (LOW-RAM fit path) ───
# Fit VarianceThreshold on a float32 ndarray, then subset columns.
# [Colab / limited RAM]

import gc
from sklearn.feature_selection import VarianceThreshold

feat_names = df_feat.columns.tolist()
X_var = df_feat.to_numpy(dtype=np.float32, copy=True)
vt = VarianceThreshold(threshold=VARIANCE_THRESHOLD)
vt.fit(X_var)
mask = vt.get_support()
del X_var
gc.collect()

dropped = [c for c, ok in zip(feat_names, mask) if not ok]
keep = [c for c, ok in zip(feat_names, mask) if ok]
df_feat = df_feat[keep]
del vt
gc.collect()

print(f'Variance threshold      : {VARIANCE_THRESHOLD}')
print(f'Near-zero var dropped   : {len(dropped)}')
print(f'  {dropped}')
print(f'Features remaining      : {df_feat.shape[1]}')

Variance threshold      : 0.01
Near-zero var dropped   : 2
  ['Subflow Bwd Packets', 'Proto_0']
Features remaining      : 77


In [ ]:
# ── Drop highly correlated features (LOW-RAM: float32 sample + cleanup) ───
# Network flow features are often redundant (e.g. max ↔ mean ↔ total bytes).
# Uses a random sample for speed — correlation estimate is accurate enough.
# [Colab / limited RAM]

import gc

n_s = min(100_000, len(df_feat))
sample = df_feat.sample(n=n_s, random_state=42).astype(np.float32, copy=False)
corr = sample.corr(numeric_only=True).abs()
upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))

drop_corr = [c for c in upper.columns if any(upper[c] > CORR_THRESHOLD)]
df_feat = df_feat.drop(columns=drop_corr, errors='ignore')

del sample, corr, upper
gc.collect()

print(f'Correlation threshold   : {CORR_THRESHOLD}')
print(f'Correlated cols dropped : {len(drop_corr)}')
print(f'  {drop_corr}')
print(f'Features remaining      : {df_feat.shape[1]}')

Correlation threshold   : 0.95
Correlated cols dropped : 23
  ['Total Bwd packets', 'Bwd Packet Length Std', 'Fwd IAT Total', 'Fwd IAT Max', 'Bwd IAT Total', 'Bwd PSH Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Packet Length Std', 'PSH Flag Count', 'ACK Flag Count', 'Average Packet Size', 'Fwd Segment Size Avg', 'Bwd Segment Size Avg', 'Subflow Fwd Bytes', 'Subflow Bwd Bytes', 'Fwd Act Data Pkts', 'Idle Mean', 'Idle Max', 'Idle Min', 'Proto_17']
Features remaining      : 54


In [ ]:
# ── Clip outliers (LOW-RAM: per-column clip) ───────────────────────────────
# Two strategies:
#   • IQR > 0  → ±3×IQR  (standard outlier removal)
#   • IQR = 0  → 99th-percentile cap  (zero-inflated features like Total Bwd Bytes)
#     Without this, extreme attack values (e.g. 22M bytes) survive scaling
#     and make reconstruction error indistinguishable between benign and attacks.

import gc

n_iqr_clip = 0
n_p99_clip = 0

for col in df_feat.columns:
    s = df_feat[col]
    if s.nunique(dropna=False) <= 1:
        continue
    q1  = float(s.quantile(0.25))
    q3  = float(s.quantile(0.75))
    iqr = q3 - q1
    if iqr > 0:
        df_feat[col] = s.clip(q1 - 3 * iqr, q3 + 3 * iqr)
        n_iqr_clip += 1
    else:
        # Zero-inflated: IQR=0 means ±3×IQR does nothing — cap at 99th percentile instead
        p99 = float(s.quantile(0.99))
        if p99 > 0:
            df_feat[col] = s.clip(upper=p99)
            n_p99_clip += 1

gc.collect()

print(f'Clipped (±3×IQR)      : {n_iqr_clip} features')
print(f'Clipped (p99 cap)     : {n_p99_clip} zero-inflated features')
print(f'Skipped (binary/const): {df_feat.shape[1] - n_iqr_clip - n_p99_clip} features')

Clipped (±3×IQR)      : 27 features
Clipped (p99 cap)     : 25 zero-inflated features
Skipped (binary/const): 2 features


## 8. Feature Scaling — StandardScaler
`StandardScaler` centers each feature to zero mean and unit variance. Unlike `QuantileTransformer` (rank-based), it **preserves relative magnitude differences** — attack-specific extreme values remain statistically large after scaling, which is essential for reconstruction-error anomaly detection. Zero-inflated features are handled by the p99 clipping step above.

In [ ]:
# [LOW-RAM] One float32 matrix, then delete df_feat before building scaled frame.
import gc

_cols = df_feat.columns.tolist()
_idx  = df_feat.index
X_num = df_feat.to_numpy(dtype=np.float32, copy=True)
del df_feat
gc.collect()

scaler = StandardScaler()
scaled = scaler.fit_transform(X_num).astype(np.float32, copy=False)
del X_num
gc.collect()

df_scaled = pd.DataFrame(scaled, columns=_cols, index=_idx)
del scaled
gc.collect()

print(f'Scaled shape : {df_scaled.shape}')
print('\nSample stats after scaling (mean≈0, std≈1):')
print(df_scaled.describe().T[['mean', 'std', '50%', 'min', 'max']].round(3).head(10))

Scaled shape : (3191164, 54)

Sample stats after scaling (mean≈0, std≈1):
                            mean    std    50%    min    max
Src Port                     0.0  0.999  0.174 -2.185  1.595
Dst Port                    -0.0  1.024  0.261 -4.019  0.796
Flow Duration               -0.0  1.003 -0.502 -0.709  1.909
Total Fwd Packet             0.0  0.998 -0.292 -2.242  1.983
Total Length of Fwd Packet   0.0  1.006 -0.411 -0.961  2.017
Total Length of Bwd Packet  -0.0  0.991 -0.149 -0.150  7.446
Fwd Packet Length Max        0.0  0.997  0.031 -1.534  2.067
Fwd Packet Length Min        0.0  0.991 -0.142 -0.142  7.307
Fwd Packet Length Mean      -0.0  0.992 -0.336 -1.089  2.347
Fwd Packet Length Std       -0.0  0.999 -0.067 -1.333  2.286


In [ ]:
# ── Save scaler ────────────────────────────────────────────────────────────
# The same scaler must be applied to new flows at inference time.
# Load it with: scaler = joblib.load('robust_scaler.pkl')

import joblib

scaler_path = f'{OUTPUT_DIR}/robust_scaler.pkl'
joblib.dump(scaler, scaler_path)
print(f'Scaler saved : {scaler_path}')

Scaler saved : /content/data/processed/robust_scaler.pkl


## 9. Final Assembly
Reattach `Src IP`, `session_id`, `Label` to the scaled feature matrix.

In [ ]:
df_processed = df_scaled.copy()
df_processed['Src IP']     = meta['Src IP'].values
df_processed['session_id'] = meta['session_id'].values
df_processed['Label']      = meta['Label'].values
df_processed['ts']         = meta['ts'].values

print(f'Final shape  : {df_processed.shape}')
print(f'Feature cols : {df_scaled.shape[1]}')
print(f'Meta cols    : Src IP, session_id, Label')
df_processed.head(3)

Final shape  : (3191164, 58)
Feature cols : 54
Meta cols    : Src IP, session_id, Label


,Src Port,Dst Port,Flow Duration,Total Fwd Packet,Total Length of Fwd Packet,Total Length of Bwd Packet,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,Active Std,Active Max,Active Min,Idle Std,Total TCP Flow Time,Proto_6,Src IP,session_id,Label,ts
0,0.498447,-3.991047,-0.702208,-1.916904,-0.920875,-0.149369,-1.419584,7.306583,-0.762353,-1.33324,...,-0.129526,-0.143111,-0.110386,-0.126764,-0.709233,-6.649896,10.16.0.4,10.16.0.4_sess0,5,2023-03-09 01:01:36.486196
1,0.130265,-3.991047,-0.704304,-1.916904,-0.902902,-0.149642,-1.368222,7.306583,-0.616350,-1.33324,...,-0.129526,-0.143111,-0.110386,-0.126764,-0.709233,-6.649896,10.16.0.4,10.16.0.4_sess1,6,2023-03-09 01:38:35.203120
2,-0.127744,-3.991047,-0.703319,-1.916904,-0.902902,-0.149642,-1.368222,7.306583,-0.616350,-1.33324,...,-0.129526,-0.143111,-0.110386,-0.126764,-0.709233,-6.649896,10.16.0.4,10.16.0.4_sess1,6,2023-03-09 01:38:59.914730


## 10. Session Window Feasibility Check
Verify each session has enough flows to produce at least one `WINDOW_SIZE` window.

In [ ]:
session_sizes = df_processed.groupby('session_id').size().sort_values(ascending=False)

feasible   = session_sizes[session_sizes >= WINDOW_SIZE]
marginal   = session_sizes[(session_sizes > 0) & (session_sizes < WINDOW_SIZE)]

print(f'WINDOW_SIZE = {WINDOW_SIZE}, STRIDE = {STRIDE}')
print(f'─────────────────────────────────────────')
print(f'Total sessions          : {len(session_sizes):,}')
print(f'Feasible (>={WINDOW_SIZE} flows) : {len(feasible):,}')
print(f'Too small (<{WINDOW_SIZE} flows)  : {len(marginal):,}')
print(f'Rows in feasible sessions: {feasible.sum():,}  ({feasible.sum()/len(df_processed)*100:.1f}%)')
print(f'\nTop 15 sessions by size:')
print(session_sizes.head(15).to_string())

WINDOW_SIZE = 500, STRIDE = 100
─────────────────────────────────────────
Total sessions          : 3,043
Feasible (>=500 flows) : 280
Too small (<500 flows)  : 2,763
Rows in feasible sessions: 3,087,524  (96.8%)

Top 15 sessions by size:
session_id
100.64.0.2_sess254    240165
100.64.0.2_sess277    151707
100.64.0.2_sess261    137331
100.64.0.2_sess263    123055
100.64.0.2_sess271    115159
100.64.0.2_sess257    107786
100.64.0.2_sess274    104422
100.64.0.2_sess273     92576
100.64.0.2_sess268     79473
100.64.0.2_sess269     74572
100.64.0.2_sess223     74490
100.64.0.2_sess260     73902
100.64.0.2_sess250     73522
100.64.0.2_sess256     70513
100.64.0.2_sess267     70019


In [ ]:
# Per-container feasibility summary
print('Per-container session/window summary:')
rows = []
for container in df_processed['Src IP'].unique():
    cdf       = df_processed[df_processed['Src IP'] == container]
    c_sess    = cdf.groupby('session_id').size()
    c_feasible= c_sess[c_sess >= WINDOW_SIZE]
    n_windows = sum(max(0, (s - WINDOW_SIZE) // STRIDE + 1) for s in c_feasible)
    rows.append({
        'Src IP'          : container,
        'total_flows'     : len(cdf),
        'sessions'        : len(c_sess),
        'feasible_sessions': len(c_feasible),
        'est_windows'     : n_windows,
        'benign_%'        : round((cdf['Label']==0).mean()*100, 1),
        'attack_%'        : round((cdf['Label']!=0).mean()*100, 1)
    })

feas_df = pd.DataFrame(rows).set_index('Src IP').sort_values('total_flows', ascending=False)
print(feas_df.to_string())

Per-container session/window summary:
            total_flows  sessions  feasible_sessions  est_windows  benign_%  attack_%
Src IP                                                                               
100.64.0.2      2822553       278                232        27118      92.0       8.0
10.16.0.9        225405        57                 31         2117     100.0       0.0
10.16.0.6         53267       503                  8          194      76.5      23.5
10.16.0.5         34400       584                  5          103      68.8      31.2
10.16.0.4         34224       577                  4          107      69.0      31.0
10.16.0.61        21315      1044                  0            0     100.0       0.0


## 11. Save Processed Data

In [ ]:
# Save full processed dataset (parquet — faster & smaller than CSV)
out_path = f'{OUTPUT_DIR}/mdc_preprocessed.parquet'
df_processed.to_parquet(out_path, index=False)
file_mb = os.path.getsize(out_path) / 1024 / 1024

print(f'Saved  : {out_path}')
print(f'Size   : {file_mb:.1f} MB')
print(f'Shape  : {df_processed.shape}')

Saved  : /content/data/processed/mdc_preprocessed.parquet
Size   : 194.6 MB
Shape  : (3191164, 58)


## 12. Preprocessing Summary

In [ ]:
print('=' * 50)
print('       PREPROCESSING SUMMARY')
print('=' * 50)
print(f'Original rows          : 3,231,475')
print(f'After container filter : {len(df_processed):,}')
print(f'Containers kept        : {df_processed["Src IP"].nunique()}')
print(f'  Min-flow threshold   : {MIN_FLOWS:,}')
print()
print(f'Sessions total         : {df_processed["session_id"].nunique()}')
print(f'  Gap threshold        : {GAP_THRESHOLD}s')
print(f'  Feasible sessions    : {len(feasible)}')
print(f'  Est. total windows   : {feas_df["est_windows"].sum():,}')
print(f'  Window size / stride : {WINDOW_SIZE} / {STRIDE}')
print()
print(f'Feature count          : {df_scaled.shape[1]}')
print(f'  Inf values fixed     : yes (→ median fill)')
print(f'  Missing values fixed : yes (→ median fill)')
print(f'  Protocol encoded     : one-hot (Proto_0 / Proto_6 / Proto_17)')
print(f'  Near-zero var drop   : yes (threshold={VARIANCE_THRESHOLD})')
print(f'  Corr drop            : yes (threshold={CORR_THRESHOLD})')
print(f'  Outlier clip         : ±3×IQR (IQR>0) | p99 cap (zero-inflated)')
print(f'  Scaling              : StandardScaler (mean=0, std=1)')
print()
print(f'Label distribution:')
label_vc = df_processed['Label'].value_counts().sort_index()
for lbl, cnt in label_vc.items():
    print(f'  Label {lbl:<3}: {cnt:>10,}  ({cnt/len(df_processed)*100:.2f}%)')
print('=' * 50)

       PREPROCESSING SUMMARY
Original rows          : 3,231,475
After container filter : 3,191,164
Containers kept        : 6
  Min-flow threshold   : 10,000

Sessions total         : 3043
  Gap threshold        : 60s
  Feasible sessions    : 280
  Est. total windows   : 29,639
  Window size / stride : 500 / 100

Feature count          : 54
  Inf values fixed     : yes (→ median fill)
  Missing values fixed : yes (→ median fill)
  Protocol encoded     : one-hot (Proto_0 / Proto_6 / Proto_17)
  Near-zero var drop   : yes (threshold=0.01)
  Corr drop            : yes (threshold=0.95)
  Outlier clip         : ±3×IQR (IQR>0) | p99 cap (zero-inflated)
  Scaling              : StandardScaler (mean=0, std=1)

Label distribution:
  Label 0  :  2,932,749  (91.90%)
  Label 1  :     92,062  (2.88%)
  Label 2  :    156,489  (4.90%)
  Label 3  :        120  (0.00%)
  Label 4  :        121  (0.00%)
  Label 5  :         14  (0.00%)
  Label 6  :         93  (0.00%)
  Label 7  :        100  (0.00%)
  L

---

## Part 2 — Bucketing, sliding windows & train/val/test splits

Reads **`mdc_preprocessed.parquet`** from `OUTPUT_DIR` (written above). Keep the **same kernel** so variables such as `OUTPUT_DIR` stay valid.

**Steps:** load parquet → 15 s buckets → zero-fill → sliding windows → temporal split → save `windows.npz`.


In [ ]:
# Part 2 — reuse OUTPUT_DIR from Part 1

INPUT_PATH = os.path.join(OUTPUT_DIR, 'mdc_preprocessed.parquet')

# ── Bucketing ─────────────────────────────────────────────────────────────
BUCKET_FREQ = '15s'   # aggregate flows into 15-second time buckets

# ── Sliding window ────────────────────────────────────────────────────────
WINDOW_SIZE = 10      # buckets per window  (10 × 15s = 150s = 2.5 min)
STRIDE      = 2       # stride in buckets   ( 2 × 15s =  30s)

# ── Split ratios ───────────────────────────────────────────────────────────
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15

# ── Autoencoder: train on benign only ─────────────────────────────────────
BENIGN_LABEL = 0

print('Part 2 config (window pipeline)')
print(f'  INPUT_PATH  = {INPUT_PATH}')
print(f'  OUTPUT_DIR  = {OUTPUT_DIR}')
print(f'  BUCKET_FREQ = {BUCKET_FREQ}')
print(f'  WINDOW_SIZE = {WINDOW_SIZE} buckets  ({WINDOW_SIZE * 15}s per window)')
print(f'  STRIDE      = {STRIDE} buckets  ({STRIDE * 15}s)')
print(f'  Split       = {TRAIN_RATIO} / {VAL_RATIO} / {TEST_RATIO}  (train/val/test)')
print(f'  Training    = benign only  (Label == {BENIGN_LABEL})')

Part 2 config (window pipeline)
  INPUT_PATH  = /content/data/processed/mdc_preprocessed.parquet
  OUTPUT_DIR  = /content/data/processed
  BUCKET_FREQ = 15s
  WINDOW_SIZE = 10 buckets  (150s per window)
  STRIDE      = 2 buckets  (30s)
  Split       = 0.7 / 0.15 / 0.15  (train/val/test)
  Training    = benign only  (Label == 0)


## 1. Load Preprocessed Data

In [ ]:
df = pd.read_parquet(INPUT_PATH)
df['ts'] = pd.to_datetime(df['ts'])

# Feature columns: everything except metadata
META_COLS    = ['Src IP', 'session_id', 'Label', 'ts']
feature_cols = [c for c in df.columns if c not in META_COLS]

print(f'Loaded shape  : {df.shape}')
print(f'Features      : {len(feature_cols)}')
print(f'Sessions      : {df["session_id"].nunique():,}')
print(f'Containers    : {df["Src IP"].nunique()}')
print(f'\nLabel counts:')
print(df['Label'].value_counts().sort_index().to_string())

Loaded shape  : (3191164, 58)
Features      : 54
Sessions      : 3,043
Containers    : 6

Label counts:
Label
0     2932749
1       92062
2      156489
3         120
4         121
5          14
6          93
7         100
8         635
9          25
10         37
11       8719


## 2. 15-Second Time Bucketing
Group flows within each session into 15-second bins.  
**Features** → `mean`, `max`, and `std` per bin — mean captures average behaviour, max captures burst peaks, std captures variability. All three are critical: attacks often appear as sudden bursts invisible to mean-only aggregation.  
**Label** → any-attack (if any flow in bin is attack → bin label = 1).

In [ ]:
# Floor each flow's timestamp to the nearest 15s bucket
df['bucket'] = df['ts'].dt.floor(BUCKET_FREQ)

# Aggregate features: mean + max + std per (container, session, bucket)
# Single groupby pass — 3× more features but captures burst signatures attacks produce
feat_bucket = (
    df.groupby(['Src IP', 'session_id', 'bucket'])[feature_cols]
    .agg(['mean', 'max', 'std'])
)
feat_bucket.columns = [f'{col}_{stat}' for col, stat in feat_bucket.columns]
feat_bucket = feat_bucket.reset_index()

# Buckets with a single flow have NaN std (no variability to measure) — fill with 0
std_cols = [c for c in feat_bucket.columns if c.endswith('_std')]
feat_bucket[std_cols] = feat_bucket[std_cols].fillna(0.0)

# Aggregate label: any-attack strategy
label_bucket = (
    df.groupby(['Src IP', 'session_id', 'bucket'])['Label']
    .apply(lambda x: 0 if (x == BENIGN_LABEL).all() else 1)
    .reset_index(name='label')
)

# Merge features and labels
bucket_df = feat_bucket.merge(label_bucket, on=['Src IP', 'session_id', 'bucket'])

# Feature columns for windowing (excludes metadata and label)
bucket_feature_cols = [
    c for c in bucket_df.columns
    if c not in ['Src IP', 'session_id', 'bucket', 'label']
]

n_buckets = len(bucket_df)
n_attack  = bucket_df['label'].sum()
print(f'Total buckets    : {n_buckets:,}')
print(f'Attack buckets   : {n_attack:,}  ({n_attack/n_buckets*100:.2f}%)')
print(f'Benign buckets   : {n_buckets - n_attack:,}')
print(f'Features/bucket  : {len(bucket_feature_cols)}  ({len(feature_cols)} original × 3 stats: mean/max/std)')
print(f'\nBuckets per session (before zero-fill):')
print(bucket_df.groupby('session_id').size().describe().round(0))

Total buckets    : 78,117
Attack buckets   : 25,469  (32.60%)
Benign buckets   : 52,648
Features/bucket  : 162  (54 original × 3 stats: mean/max/std)

Buckets per session (before zero-fill):
count    3043.0
mean       26.0
std       205.0
min         1.0
25%         1.0
50%         2.0
75%        32.0
max      5760.0
dtype: float64


## 3. Zero-Fill Empty Buckets
Within each session, create a complete 15s time range.  
Buckets with no traffic → filled with **zeros** (silence = signal for the autoencoder).

In [ ]:
def zero_fill_session(group):
    """Extend session to a complete 15s grid, fill missing buckets with 0."""
    src_ip  = group['Src IP'].iloc[0]
    sess_id = group['session_id'].iloc[0]

    full_range = pd.date_range(
        start=group['bucket'].min(),
        end=group['bucket'].max(),
        freq=BUCKET_FREQ
    )

    group = (
        group
        .set_index('bucket')
        .drop(columns=['Src IP', 'session_id'])
        .reindex(full_range, fill_value=0.0)
    )
    group['Src IP']     = src_ip
    group['session_id'] = sess_id
    group['label']      = group['label'].astype(int)

    return group.reset_index().rename(columns={'index': 'bucket'})

before_count = len(bucket_df)

bucket_df = (
    bucket_df
    .groupby(['Src IP', 'session_id'], group_keys=False)
    .apply(zero_fill_session)
    .reset_index(drop=True)
)

after_count = len(bucket_df)
print(f'Buckets before fill : {before_count:,}')
print(f'Buckets after  fill : {after_count:,}')
print(f'Zero-filled added   : {after_count - before_count:,}')
print(f'\nBuckets per session (after zero-fill):')
print(bucket_df.groupby('session_id').size().describe().round(0))

Buckets before fill : 78,117
Buckets after  fill : 86,640
Zero-filled added   : 8,523

Buckets per session (after zero-fill):
count    3043.0
mean       28.0
std       220.0
min         1.0
25%         1.0
50%         2.0
75%        40.0
max      5760.0
dtype: float64


## 4. Sliding Window Construction
Slide a window of `WINDOW_SIZE` buckets with `STRIDE` over each session.  
Windows **never cross session boundaries**.  
**Window label** → any-attack (1 if any bucket in window is attack).

In [ ]:
def make_windows(group_df, W, S, feat_cols):
    """Slide window of W buckets with stride S over one session."""
    arr    = group_df[feat_cols].values.astype(np.float32)  # (n_buckets, n_features)
    labels = group_df['label'].values
    n      = len(arr)

    wins, lbls = [], []
    for start in range(0, n - W + 1, S):
        wins.append(arr[start : start + W])
        # Any-attack: 1 if any bucket in window is attack
        lbls.append(0 if (labels[start : start + W] == 0).all() else 1)

    return wins, lbls

all_windows, all_labels, all_sessions = [], [], []

for (src_ip, sess_id), grp in bucket_df.groupby(['Src IP', 'session_id']):
    grp  = grp.sort_values('bucket')
    wins, lbls = make_windows(grp, WINDOW_SIZE, STRIDE, bucket_feature_cols)
    all_windows.extend(wins)
    all_labels.extend(lbls)
    all_sessions.extend([sess_id] * len(wins))

X_all = np.array(all_windows, dtype=np.float32)  # (N, WINDOW_SIZE, n_features)
y_all = np.array(all_labels,  dtype=np.int8)      # (N,)

print(f'Total windows    : {len(X_all):,}')
print(f'Shape            : {X_all.shape}   (windows × buckets × features)')
print(f'Attack windows   : {y_all.sum():,}  ({y_all.mean()*100:.2f}%)')
print(f'Benign windows   : {(y_all == 0).sum():,}  ({(y_all == 0).mean()*100:.2f}%)')

Total windows    : 36,810
Shape            : (36810, 10, 162)   (windows × buckets × features)
Attack windows   : 13,647  (37.07%)
Benign windows   : 23,163  (62.93%)


## 5. Hybrid Train / Val / Test Split

**Strategy for anomaly detection:**
1. **Train:** First 70% of sessions (temporal) → filtered to **benign-only** for autoencoder
2. **Val + Test:** Remaining 30% → **stratified split** ensuring both contain attacks

This hybrid approach:
- Preserves temporal ordering for training (no future leakage)  
- Ensures validation has attack samples for **threshold tuning** (critical!)
- Ensures test has attack samples for **final evaluation**

In [ ]:
# Sort sessions by their earliest bucket timestamp
session_start = (
    bucket_df.groupby('session_id')['bucket']
    .min()
    .sort_values()
)
sessions_ordered = session_start.index.tolist()
n_sess = len(sessions_ordered)

# ═══════════════════════════════════════════════════════════════════════════
# STEP 1: Temporal split for TRAIN (first 70% sessions)
# ═══════════════════════════════════════════════════════════════════════════
train_end = int(n_sess * TRAIN_RATIO)
train_sessions = set(sessions_ordered[:train_end])
holdout_sessions = set(sessions_ordered[train_end:])  # remaining 30% for val+test

train_mask   = np.array([s in train_sessions   for s in all_sessions])
holdout_mask = np.array([s in holdout_sessions for s in all_sessions])

X_train_all, y_train_all = X_all[train_mask], y_all[train_mask]
X_holdout,   y_holdout   = X_all[holdout_mask], y_all[holdout_mask]

# Filter train to benign only — autoencoder learns normal patterns
benign_mask = (y_train_all == BENIGN_LABEL)
X_train     = X_train_all[benign_mask]
y_train     = y_train_all[benign_mask]   # all zeros

# ═══════════════════════════════════════════════════════════════════════════
# STEP 2: Stratified split of HOLDOUT → VAL + TEST (ensures attacks in both)
# ═══════════════════════════════════════════════════════════════════════════
from sklearn.model_selection import train_test_split

# Val gets 50% of holdout (≈15% of total), Test gets 50% (≈15% of total)
# stratify=y_holdout ensures attack ratio is balanced between val and test
X_val, X_test, y_val, y_test = train_test_split(
    X_holdout, y_holdout,
    test_size=0.5,
    stratify=y_holdout,
    random_state=42
)

# ═══════════════════════════════════════════════════════════════════════════
# Summary
# ═══════════════════════════════════════════════════════════════════════════
print(f'Sessions  →  train: {len(train_sessions):,}  |  holdout (val+test): {len(holdout_sessions):,}')
print()
print(f'X_train (benign only) : {X_train.shape}')
print(f'  Discarded attack    : {(~benign_mask).sum():,} windows  (not used in training)')
print()
print(f'X_val   : {X_val.shape}   attack: {y_val.sum():,} ({y_val.mean()*100:.1f}%)')
print(f'X_test  : {X_test.shape}  attack: {y_test.sum():,} ({y_test.mean()*100:.1f}%)')
print()
print('✓ Validation now has attacks for threshold tuning!')

Sessions  →  train: 2,130  |  holdout (val+test): 913

X_train (benign only) : (15826, 10, 162)
  Discarded attack    : 83 windows  (not used in training)

X_val   : (10450, 10, 162)   attack: 6,782 (64.9%)
X_test  : (10451, 10, 162)  attack: 6,782 (64.9%)

✓ Validation now has attacks for threshold tuning!


## 6. Save

In [ ]:
out_path = f'{OUTPUT_DIR}/windows.npz'

np.savez_compressed(
    out_path,
    X_train = X_train,   # (N_train, W, n_features) — benign only
    X_val   = X_val,     # (N_val,   W, n_features) — all labels
    X_test  = X_test,    # (N_test,  W, n_features) — all labels
    y_val   = y_val,     # binary: 0=benign, 1=attack
    y_test  = y_test
)

file_mb = os.path.getsize(out_path) / 1024 / 1024

print(f'Saved  : {out_path}')
print(f'Size   : {file_mb:.1f} MB')
print()
print('=' * 50)
print('           DATALOADER SUMMARY')
print('=' * 50)
print(f'Bucket size    : {BUCKET_FREQ}')
print(f'Window size    : {WINDOW_SIZE} buckets  ({WINDOW_SIZE*15}s)')
print(f'Stride         : {STRIDE} buckets  ({STRIDE*15}s)')
print(f'Total windows  : {len(X_all):,}')
print()
print(f'X_train        : {X_train.shape}  ← benign only')
print(f'X_val          : {X_val.shape}')
print(f'X_test         : {X_test.shape}')
print(f'y_val          : {y_val.shape}')
print(f'y_test         : {y_test.shape}')
print('=' * 50)

Saved  : /content/data/processed/windows.npz
Size   : 41.4 MB

           DATALOADER SUMMARY
Bucket size    : 15s
Window size    : 10 buckets  (150s)
Stride         : 2 buckets  (30s)
Total windows  : 36,810

X_train        : (15826, 10, 162)  ← benign only
X_val          : (10450, 10, 162)
X_test         : (10451, 10, 162)
y_val          : (10450,)
y_test         : (10451,)


## 7. Copy artifacts to Google Drive (Colab)

Persists **`windows.npz`**, **`mdc_preprocessed.parquet`**, and **`robust_scaler.pkl`** under **`My Drive/Module4_MDC/processed/`** (change `DRIVE_EXPORT_SUBDIR` if you want). On a local machine this cell **skips** mount/copy — upload the same three files to Drive manually, or sync `data/processed/`.

In [ ]:
import json
import shutil
from datetime import datetime, timezone
from pathlib import Path

# Folder *under* Google Drive "My Drive" (no leading slash)
DRIVE_EXPORT_SUBDIR = 'Module4_MDC/processed'

ARTIFACTS = [
    'windows.npz',
    'mdc_preprocessed.parquet',
    'robust_scaler.pkl',
]

def export_to_google_drive():
    if not _IN_COLAB:
        print('Not on Colab — skip Drive export. Copy from OUTPUT_DIR manually:')
        print(' ', OUTPUT_DIR)
        return
    from google.colab import drive

    import os as _os
    _dr = '/content/drive'
    _md = _os.path.join(_dr, 'MyDrive')
    if _os.path.isdir(_md):
        print('Google Drive already mounted:', _md)
    else:
        drive.mount(_dr, force_remount=False)
    dst_root = Path('/content/drive/MyDrive') / DRIVE_EXPORT_SUBDIR
    dst_root.mkdir(parents=True, exist_ok=True)
    out_root = Path(OUTPUT_DIR)
    copied = []
    for name in ARTIFACTS:
        src = out_root / name
        if not src.is_file():
            print(f'[skip] missing: {src}')
            continue
        shutil.copy2(src, dst_root / name)
        copied.append(name)
        print(f'Copied → {dst_root / name}')
    manifest = {
        'exported_at_utc': datetime.now(timezone.utc).isoformat(),
        'drive_folder': str(dst_root),
        'files': copied,
        'shapes': {
            'X_train': list(X_train.shape),
            'X_val': list(X_val.shape),
            'X_test': list(X_test.shape),
        },
        'bucket_freq': BUCKET_FREQ,
        'window_buckets': int(WINDOW_SIZE),
        'stride_buckets': int(STRIDE),
    }
    man_path = dst_root / 'export_manifest.json'
    man_path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
    print(f'Wrote manifest: {man_path}')
    print('\nNext notebook: set DRIVE_DATA_DIR to this folder (see mdc_model_training.ipynb).')


export_to_google_drive()